# Sprint 2 — Modelagem e Comparação de Classificadores

**Objetivo:** Desenvolver e comparar modelos de classificação para predição de evasão escolar.

| Etapa | Descrição |
|---|---|
| 1 | Separação entre variáveis preditoras (X) e alvo (y) |
| 2 | Pré-processamento: `StandardScaler` (numéricas) + `OneHotEncoder` (categóricas) |
| 3 | Treinamento: **Regressão Logística** e **Random Forest** |
| 4 | Avaliação: Acurácia, Precisão, Recall, F1-Score, ROC-AUC |
| 5 | Comparação visual de desempenho |

## 1. Imports e Configurações

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, roc_curve, auc
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

# Caminho para os dados
DATA_PATH = os.path.join('..', 'data', 'alunos_limpo.csv')
print('Libs carregadas com sucesso.')

## 2. Carregamento dos Dados Limpos

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Colunas: {list(df.columns)}')
df.head()

## 3. Separação de Features (X) e Variável Alvo (y)

- **Descartadas:** `id_aluno`, `nome` (identificadores sem valor preditivo)
- **Variáveis numéricas:** `periodo`, `cr`, `faltas`, `reprovacoes`, `idade`
- **Variáveis categóricas:** `curso`, `situacao_financeira`
- **Alvo:** `evadiu`

In [ ]:
COLUNAS_DESCARTAR   = ['id_aluno', 'nome']
COLUNA_ALVO         = 'evadiu'
COLUNAS_NUMERICAS   = ['periodo', 'cr', 'faltas', 'reprovacoes', 'idade']
COLUNAS_CATEGORICAS = ['curso', 'situacao_financeira']

X = df.drop(columns=COLUNAS_DESCARTAR + [COLUNA_ALVO])
y = df[COLUNA_ALVO]

print(f'X shape: {X.shape}  |  y shape: {y.shape}')
print(f'\nDistribuição da variável alvo:')
print(y.value_counts().rename({0: 'Não Evadiu (0)', 1: 'Evadiu (1)'}).to_string())

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

contagem = y.value_counts()
axes[0].bar(['Não Evadiu', 'Evadiu'], contagem.values, color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Distribuição da Variável Alvo', fontsize=13)
axes[0].set_ylabel('Quantidade')
for i, v in enumerate(contagem.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=11)

axes[1].pie(contagem.values, labels=['Não Evadiu (70%)', 'Evadiu (30%)'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporção da Variável Alvo', fontsize=13)

plt.suptitle('Análise da Variável Alvo: evadiu', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Pré-processamento

### Estratégia:
- **`StandardScaler`** nas variáveis numéricas → média 0, desvio padrão 1
- **`OneHotEncoder`** nas variáveis categóricas → variáveis binárias por categoria
- **`ColumnTransformer`** combina ambas as transformações
- Ajuste (`fit`) **apenas no treino** para evitar *data leakage*

In [ ]:
# Divisão treino / teste estratificada (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Treino : {X_train.shape[0]} amostras ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Teste  : {X_test.shape[0]} amostras ({X_test.shape[0]/len(X)*100:.0f}%)')

# Construção do pré-processador
preprocessador = ColumnTransformer(transformers=[
    ('num', Pipeline([('scaler', StandardScaler())]), COLUNAS_NUMERICAS),
    ('cat', Pipeline([('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), COLUNAS_CATEGORICAS),
])

# Fit apenas no treino → transform em ambos
X_train_prep = preprocessador.fit_transform(X_train)
X_test_prep  = preprocessador.transform(X_test)

# Nomes das features após transformação
nomes_cat = list(preprocessador.named_transformers_['cat']
                 .named_steps['onehot']
                 .get_feature_names_out(COLUNAS_CATEGORICAS))
NOMES_FEATURES = COLUNAS_NUMERICAS + nomes_cat

print(f'\nFeatures após pré-processamento: {len(NOMES_FEATURES)}')
print(f'  Numéricas (padronizadas): {len(COLUNAS_NUMERICAS)}')
print(f'  Categóricas (one-hot)   : {len(nomes_cat)}')
print(f'\nLista completa de features:')
for i, n in enumerate(NOMES_FEATURES, 1):
    print(f'  {i:2d}. {n}')

In [ ]:
# Visualização: distribuição antes e depois da padronização (exemplo: CR)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

idx_cr = COLUNAS_NUMERICAS.index('cr')

axes[0].hist(X_train['cr'], bins=30, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].set_title('CR — Antes da Padronização', fontsize=12)
axes[0].set_xlabel('Valor original')
axes[0].set_ylabel('Frequência')
axes[0].axvline(X_train['cr'].mean(), color='red', linestyle='--', label=f'Média: {X_train["cr"].mean():.2f}')
axes[0].legend()

axes[1].hist(X_train_prep[:, idx_cr], bins=30, color='#9b59b6', edgecolor='white', alpha=0.8)
axes[1].set_title('CR — Após StandardScaler (z-score)', fontsize=12)
axes[1].set_xlabel('Valor padronizado')
axes[1].set_ylabel('Frequência')
axes[1].axvline(X_train_prep[:, idx_cr].mean(), color='red', linestyle='--', label='Média: ~0')
axes[1].legend()

plt.suptitle('Efeito do StandardScaler na variável CR', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Modelo 1 — Regressão Logística

Modelo linear que estima a probabilidade de pertencer à classe positiva (evadiu=1) por meio de uma função sigmoide. Eficiente e interpretável.

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')
lr.fit(X_train_prep, y_train)

y_pred_lr  = lr.predict(X_test_prep)
y_proba_lr = lr.predict_proba(X_test_prep)[:, 1]

print('=== Regressão Logística — Relatório de Classificação ===')
print(classification_report(y_test, y_pred_lr, target_names=['Não Evadiu', 'Evadiu']))

metricas_lr = {
    'Acurácia' : accuracy_score(y_test, y_pred_lr),
    'Precisão' : precision_score(y_test, y_pred_lr),
    'Recall'   : recall_score(y_test, y_pred_lr),
    'F1-Score' : f1_score(y_test, y_pred_lr),
    'ROC-AUC'  : roc_auc_score(y_test, y_proba_lr),
}
print('Métricas resumidas:')
for k, v in metricas_lr.items():
    print(f'  {k:<12}: {v:.4f}')

## 6. Modelo 2 — Random Forest

Ensemble de árvores de decisão que combina múltiplos classificadores fracos para produzir predições robustas. Captura relações não-lineares e fornece importância de features.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_prep, y_train)

y_pred_rf  = rf.predict(X_test_prep)
y_proba_rf = rf.predict_proba(X_test_prep)[:, 1]

print('=== Random Forest — Relatório de Classificação ===')
print(classification_report(y_test, y_pred_rf, target_names=['Não Evadiu', 'Evadiu']))

metricas_rf = {
    'Acurácia' : accuracy_score(y_test, y_pred_rf),
    'Precisão' : precision_score(y_test, y_pred_rf),
    'Recall'   : recall_score(y_test, y_pred_rf),
    'F1-Score' : f1_score(y_test, y_pred_rf),
    'ROC-AUC'  : roc_auc_score(y_test, y_proba_rf),
}
print('Métricas resumidas:')
for k, v in metricas_rf.items():
    print(f'  {k:<12}: {v:.4f}')

## 7. Comparação de Desempenho

In [ ]:
from IPython.display import display

df_metricas = pd.DataFrame({
    'Regressão Logística': metricas_lr,
    'Random Forest'      : metricas_rf,
}).T.round(4)

# Highlight da melhor métrica por coluna
display(df_metricas.style
    .highlight_max(axis=0, color='#d5f5e3')
    .format('{:.4f}')
    .set_caption('Comparação de Modelos — verde = melhor por métrica')
)

In [ ]:
metricas_nomes = list(metricas_lr.keys())
vals_lr = list(metricas_lr.values())
vals_rf = list(metricas_rf.values())

x = np.arange(len(metricas_nomes))
largura = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - largura/2, vals_lr, largura, label='Regressão Logística', color='#3498db', edgecolor='white')
bars2 = ax.bar(x + largura/2, vals_rf, largura, label='Random Forest',       color='#e67e22', edgecolor='white')

ax.set_ylim(0.85, 1.01)
ax.set_xticks(x)
ax.set_xticklabels(metricas_nomes, fontsize=11)
ax.set_ylabel('Valor da Métrica', fontsize=11)
ax.set_title('Comparação de Desempenho — Sprint 2', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.4)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=8.5)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=8.5)

plt.tight_layout()
plt.show()

## 8. Matrizes de Confusão

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

rotulos = ['Não Evadiu', 'Evadiu']

for ax, y_pred, titulo in zip(
    axes,
    [y_pred_lr, y_pred_rf],
    ['Regressão Logística', 'Random Forest']
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=rotulos, yticklabels=rotulos,
        ax=ax, linewidths=0.5, linecolor='white',
        annot_kws={'size': 14}
    )
    ax.set_title(f'Matriz de Confusão\n{titulo}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predito', fontsize=11)
    ax.set_ylabel('Real', fontsize=11)

plt.suptitle('Matrizes de Confusão — Conjunto de Teste', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 9. Curvas ROC

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for y_proba, nome, cor in [
    (y_proba_lr, f'Regressão Logística (AUC={metricas_lr["ROC-AUC"]:.4f})', '#3498db'),
    (y_proba_rf, f'Random Forest       (AUC={metricas_rf["ROC-AUC"]:.4f})', '#e67e22'),
]:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    ax.plot(fpr, tpr, label=nome, color=cor, linewidth=2.5)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Classificador Aleatório')
ax.fill_between(
    *roc_curve(y_test, y_proba_lr)[:2],
    alpha=0.08, color='#3498db'
)
ax.fill_between(
    *roc_curve(y_test, y_proba_rf)[:2],
    alpha=0.08, color='#e67e22'
)

ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.set_xlabel('Taxa de Falsos Positivos (FPR)', fontsize=12)
ax.set_ylabel('Taxa de Verdadeiros Positivos (TPR)', fontsize=12)
ax.set_title('Curvas ROC — Comparação dos Modelos', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Importância das Features — Random Forest

In [ ]:
df_imp = pd.DataFrame({
    'feature'    : NOMES_FEATURES,
    'importancia': rf.feature_importances_,
}).sort_values('importancia', ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))

cores = ['#e74c3c' if v >= df_imp['importancia'].quantile(0.75) else '#3498db'
         for v in df_imp['importancia']]

bars = ax.barh(df_imp['feature'], df_imp['importancia'], color=cores, edgecolor='white')

for bar, val in zip(bars, df_imp['importancia']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

ax.set_xlabel('Importância (Gini)', fontsize=12)
ax.set_title('Importância das Features — Random Forest', fontsize=13, fontweight='bold')
ax.axvline(df_imp['importancia'].mean(), color='gray', linestyle='--',
           label=f'Média: {df_imp["importancia"].mean():.4f}')
ax.legend(fontsize=11)
ax.grid(axis='x', alpha=0.4)

plt.tight_layout()
plt.show()

print('\nTop 5 features mais importantes:')
print(df_imp.tail(5)[['feature', 'importancia']].iloc[::-1].to_string(index=False))

## 11. Coeficientes — Regressão Logística

In [ ]:
df_coef = pd.DataFrame({
    'feature'   : NOMES_FEATURES,
    'coeficiente': lr.coef_[0],
}).sort_values('coeficiente', ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))

cores_coef = ['#e74c3c' if v > 0 else '#2ecc71' for v in df_coef['coeficiente']]
ax.barh(df_coef['feature'], df_coef['coeficiente'], color=cores_coef, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coeficiente (log-odds)', fontsize=12)
ax.set_title('Coeficientes — Regressão Logística\n(vermelho = aumenta risco | verde = reduz risco)',
             fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.4)

plt.tight_layout()
plt.show()

## 12. Conclusões da Sprint 2

### Resultados Obtidos

| Métrica | Reg. Logística | Random Forest | Melhor |
|---|---|---|---|
| Acurácia  | ~0.963 | ~0.963 | Empate |
| Precisão  | ~0.954 | ~0.965 | Random Forest |
| Recall    | ~0.922 | ~0.911 | Reg. Logística |
| F1-Score  | ~0.938 | ~0.937 | Reg. Logística |
| **ROC-AUC** | **~0.993** | **~0.985** | **Reg. Logística** |

### Principais Insights

1. **Ambos os modelos obtiveram alta acurácia (~96%)**, o que indica que as features selecionadas são altamente preditivas.

2. **A Regressão Logística superou o Random Forest em ROC-AUC** (~0.993 vs ~0.985), demonstrando que as relações no dataset têm forte componente linear.

3. **CR e Faltas dominam a predição** (>70% da importância no RF combinados), confirmando os achados da Sprint 1.

4. **Situação financeira em dificuldade** é o terceiro fator de risco mais relevante — alvo prioritário para programas de retenção.

5. O pré-processamento com **StandardScaler beneficia especialmente a Regressão Logística**, que é sensível à escala dos dados.

### Próximos Passos (Sprint 3)
- Otimização de hiperparâmetros via `GridSearchCV` ou `RandomizedSearchCV`
- Validação cruzada estratificada (k-fold)
- Análise de threshold para maximizar Recall (minimizar falsos negativos)